# Get Bioavailable Iron from USDA

In [59]:
# Parameters
SAVE_DFS = False

In [60]:
import ast
import os
import pandas as pd
import requests

from dotenv import load_dotenv
from IPython.display import Image
from pathlib import Path
from sklearn.model_selection import train_test_split
from tqdm import tqdm

In [61]:
# Load splits from CSVs
df_train = pd.read_csv('../data/train/df_train.csv')
df_val = pd.read_csv('../data/val/df_val.csv')
df_test = pd.read_csv('../data/test/df_test.csv')

df_train.head(2)

,image_url,camera_or_phone_prob,food_prob,dish_name,food_type,ingredients,portion_size,nutritional_profile,cooking_method,sub_dt,image_name
0,https://file.b18a.io/7832973280900104501_54585...,0.8,0.90,oysters,homemade food,['oysters'],{'oysters': '500g'},"{'fat_g': 5.0, 'protein_g': 20.0, 'calories_kc...",raw,20250710,7832973280900104501_545859_.jpeg
1,https://file.b18a.io/7835136777400102715_70587...,0.7,0.95,grilled steak,restaurant food,"['steak', 'broccoli', 'potato', 'tomato', 'sau...","{'steak': '250g', 'broccoli': '50g', 'potato':...","{'fat_g': 30.0, 'protein_g': 50.0, 'calories_k...",grilling,20250702,7835136777400102715_705873_.jpeg


In [62]:
# Combine dfs
df_all = pd.concat([df_train, df_val, df_test])
df_all = df_all.reset_index()

df_all.shape

(5000, 12)

In [123]:
df_all['dish_name']

0                              oysters
1                        grilled steak
2              sweet and sour potatoes
3                              hot pot
4                   stir-fried noodles
                     ...              
4995                       noodle soup
4996              braised chicken feet
4997                           hot pot
4998    vegetable and chicken sandwich
4999                mixed asian dishes
Name: dish_name, Length: 5000, dtype: object

In [63]:
df_all['portion_size'][0]

"{'oysters': '500g'}"

In [64]:
df_all['portion_size'] = df_all['portion_size'].apply(ast.literal_eval)

df_all['portion_size'][0]

{'oysters': '500g'}

In [65]:
# Get all ingredients
ingredients = set()
for row in df_all.itertuples():
    keys = row.portion_size.keys()
    ingredients.update(row.portion_size.keys())
print(f'{len(ingredients)=}')
ingredients

len(ingredients)=822


{'abalone',
 'almonds',
 'anchovies',
 'apple',
 'apple chips',
 'apples',
 'apricot',
 'apricots',
 'asparagus',
 'assorted desserts',
 'avocado',
 'baby corn',
 'bacon',
 'bagel',
 'baked beans',
 'baked goods',
 'baked potato',
 'baklava',
 'bamboo shoots',
 'banana',
 'bananas',
 'bao dough',
 'bar',
 'base',
 'basil',
 'batter',
 'bbq ribs',
 'bean sprouts',
 'beans',
 'beef',
 'beef liver',
 'beef patty',
 'beef ribs',
 'beef stew',
 'beef tripe',
 'beef_snack',
 'beer',
 'beet',
 'beetroot',
 'beets',
 'bell pepper',
 'bell peppers',
 'berries',
 'beverage',
 'beverages',
 'bird',
 'birds',
 'biscuit',
 'biscuits',
 'bitter melon',
 'black beans',
 'black fungus',
 'black jelly',
 'black olive tapenade',
 'black pudding',
 'black rice',
 'black sapote',
 'blackberries',
 'blueberries',
 'boiled egg',
 'boiled eggs',
 'bok choy',
 'borscht',
 'bounty bar',
 'bread',
 'bread roll',
 'bread rolls',
 'bread_roll',
 'breading',
 'brisket',
 'broad beans',
 'broccoli',
 'broth',
 'buc

In [ ]:
num_ingredients = len(ingredients)
initial_vals = [0] * num_ingredients
ingredients_df = {'ingredients': list(ingredients), 'iron': initial_vals, 'calcium': initial_vals, 'vitamin_c': initial_vals}
ingredients_df = pd.DataFrame(ingredients_df)
ingredients_df


,ingredients,iron,calcium,vitamin_c
0,tuna_sushi,0,0,0
1,cucumbers,0,0,0
2,kebab,0,0,0
3,bun,0,0,0
4,energy drink,0,0,0
...,...,...,...,...
817,beets,0,0,0
818,juice,0,0,0
819,muffin,0,0,0
820,main ingredient,0,0,0


In [66]:
# Load and get api key from .env
load_dotenv()
api_key = os.getenv("API_KEY")

In [89]:
FOODS_SEARCH_URL = 'https://api.nal.usda.gov/fdc/v1/foods/search'
FOODS_DETAIL_URL = 'https://api.nal.usda.gov/fdc/v1/food/'

nutrition_data = {}
ingredients_test = ['abalone', 'chicken', 'tuna']

In [121]:
# Search for food
ingredient = 'chicken'
food_query = ingredient
r = requests.get(f'https://api.nal.usda.gov/fdc/v1/foods/search?api_key={api_key}&query={food_query}&dataType=Foundation')
data = r.json()
data_df = pd.DataFrame.from_dict(data['foods'])
if len(data_df) == 0:  # Data not found
    r = requests.get(f'https://api.nal.usda.gov/fdc/v1/foods/search?api_key={api_key}&query={food_query}')
    data = r.json()
    data_df = pd.DataFrame.from_dict(data['foods'])
data_df

,fdcId,description,commonNames,additionalDescriptions,dataType,ndbNumber,publishedDate,foodCategory,mostRecentAcquisitionDate,allHighlightFields,score,microbes,foodNutrients,finalFoodInputFoods,foodMeasures,foodAttributes,foodAttributeTypes,foodVersionIds
0,2514746,"Chicken, ground, with additives, raw",,,Foundation,5332,2023-04-20,Poultry Products,2023-01-09,,201.99571,[],"[{'nutrientId': 1089, 'nutrientName': 'Iron, F...",[],[],[],[],[]
1,2646170,"Chicken, breast, boneless, skinless, raw",,,Foundation,100304,2023-10-26,Poultry Products,2023-04-03,,187.15979,[],"[{'nutrientId': 1089, 'nutrientName': 'Iron, F...",[],[],[],[],[]
2,2727569,"Chicken, breast, meat and skin, raw",,,Foundation,100375,2025-04-24,Poultry Products,2024-10-27,,187.15979,[],"[{'nutrientId': 1089, 'nutrientName': 'Iron, F...",[],[],[],[],[]
3,2727566,"Chicken, drumstick, meat and skin, raw",,,Foundation,100372,2025-04-24,Poultry Products,2024-10-13,,187.15979,[],"[{'nutrientId': 1089, 'nutrientName': 'Iron, F...",[],[],[],[],[]
4,2646171,"Chicken, thigh, boneless, skinless, raw",,,Foundation,100305,2023-10-26,Poultry Products,2023-03-27,,187.15979,[],"[{'nutrientId': 1089, 'nutrientName': 'Iron, F...",[],[],[],[],[]
5,2727567,"Chicken, thigh, meat and skin, raw",,,Foundation,100373,2025-04-24,Poultry Products,2024-10-20,,187.15979,[],"[{'nutrientId': 1089, 'nutrientName': 'Iron, F...",[],[],[],[],[]
6,2727568,"Chicken, wing, meat and skin, raw",,,Foundation,100374,2025-04-24,Poultry Products,2024-10-27,,187.15979,[],"[{'nutrientId': 1089, 'nutrientName': 'Iron, F...",[],[],[],[],[]
7,331897,"Chicken, broilers or fryers, drumstick, meat o...",,,Foundation,5671,2019-04-01,Poultry Products,2010-05-11,,153.41048,[],"[{'nutrientId': 1253, 'nutrientName': 'Cholest...",[],[],[],[],[]
8,331960,"Chicken, broiler or fryers, breast, skinless, ...",,,Foundation,5746,2019-04-01,Poultry Products,2012-11-12,,136.97621,[],"[{'nutrientId': 1170, 'nutrientName': 'Pantoth...",[],[],[],[],[]


In [122]:
nutrients = pd.DataFrame(data_df.iloc[0]['foodNutrients'])
nutrients = nutrients.loc[nutrients['nutrientId'].isin([1087, 1089, 1162])][['nutrientName', 'value', 'unitName']]
nutrients

,nutrientName,value,unitName
0,"Iron, Fe",0.593,MG
13,"Calcium, Ca",5.820,MG


In [116]:
# Get food nutrients
if len(data_df) > 0:  # Data found
    nutrients = pd.DataFrame(data_df.iloc[0]['foodNutrients'])
    nutrients = nutrients.loc[nutrients['nutrientId'].isin([1087, 1089, 1162])]
    nutrition_data[ingredient] = {'found': True, 'vitamin_c': 0, 'iron': 0, 'calcium': 0}
else:  # Data not found
    nutrition_data[ingredient] = {'found': False, 'vitamin_c': 0, 'iron': 0, 'calcium': 0}

In [117]:
nutrients

,nutrientId,nutrientName,nutrientNumber,unitName,value,rank,indentLevel,foodNutrientId
10,1087,"Calcium, Ca",301,MG,39.00,5300,1,34198114
11,1089,"Iron, Fe",303,MG,3.97,5400,1,34198115
28,1162,"Vitamin C, total ascorbic acid",401,MG,2.00,6300,1,34198132
